# Generation files routes.txt 
A partir de rutas crudas scrapeadas de ruta dieracta, se formatean en este formato:

| route_id | agency_id | route_short_name | route_long_name | route_type |
| :--- | :--- | :--- | :--- | :--- |
| R_LINEA1 | AG_01 | L1 | Línea 1: Sur - Norte | 3 |


Tabla equivalenciar route_type, proviene del estnadar GTFS:

| Código 	| Nombre GTFS 	| Descripción                                          	|
|--------	|-------------	|------------------------------------------------------	|
| 0      	| Tram        	| Tranvía, tren ligero o Streetcar.                    	|
| 1      	| Subway      	| Metro o tren de alta capacidad (Subterráneo).        	|
| 2      	| Rail        	| Ferrocarril nacional o interurbano (ej. Tren Maya).  	|
| 3      	| Bus         	| Cualquier servicio de autobús urbano (el más común). 	|
| 4      	| Ferry       	| Transbordadores o barcos de pasajeros.               	|
| 5      	| Cable Tram  	| Tranvía de cable (estilo San Francisco).             	|
| 6      	| Aerial Lift 	| Teleféricos, Góndolas o Cablebús.                    	|
| 7      	| Funicular   	| Trenes para pendientes pronunciadas.                 	|
| 11     	| Trolleybus  	| Autobuses eléctricos con catenaria superior.         	|
| 12     	| Monorail    	| Monorriel.                                           	|

In [1]:
import pandas as pd
from pathlib import Path
import geopandas as gpd

## Parámetros

In [2]:
import json
from pathlib import Path as PathLib

_params_path = PathLib.cwd() / "params.json"
if not _params_path.exists():
    _params_path = PathLib("params.json")
with open(_params_path, encoding="utf-8") as f:
    p = json.load(f)

CIUDAD = p["ciudad"]
AGENCY_ID = p["agency"]["id"]

GTFS_ROUTE_TYPES = {
    "Bus": 3,
    "Autobús": 3,
    "Micro": 3,
    "Combi": 3,
    "Colectivo": 3,
    "Metro": 1,
    "Subway": 1,
    "Tren Ligero": 0,
    "Tram": 0,
    "Tren": 2,
    "Rail": 2,
    "Ferry": 4,
    "Barco": 4,
    "Teleférico": 6,
    "Cablebús": 6,
    "Trolebús": 11,
    "Funicular": 7
}

In [3]:
# CIUDAD y AGENCY_ID cargados desde params.json en la celda anterior

In [4]:
# --- Carpeta GTFS ---
PATH_DIR_GTFS = Path(f"../data/{CIUDAD}/gtfs-output")
PATH_DIR_GTFS.mkdir(parents=True, exist_ok=True)
print(f"Salida: {PATH_DIR_GTFS.absolute()}")

# --- Carpeta proccesed ---
PATH_DIR_proccesed = Path(f"../data/{CIUDAD}/processed")
PATH_DIR_proccesed.mkdir(parents=True, exist_ok=True)
print(f"Salida: {PATH_DIR_proccesed.absolute()}")

Salida: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generation-gtfs-from-shapes-v2/2-generation-files-simples/../data/tampico/gtfs-output
Salida: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generation-gtfs-from-shapes-v2/2-generation-files-simples/../data/tampico/processed


## Lectura y procesamiento rutas scrapeadas

### Read shapes desde geojson crudo

In [5]:
path_routes_raw = Path(f"../1-scraping_ruta_directa/data/proc/{CIUDAD}.geojson")
print(f"Ruta de shapes crudos: {path_routes_raw.absolute()}")
## Read shapes desde geojson crudo
routes_raw = gpd.read_file(path_routes_raw)
print(f"Shapes crudos (# rutas): {routes_raw.shape}")
routes_raw["route_id"] = routes_raw["slug"].str.split("-").str[1]
routes_raw.head()

Ruta de shapes crudos: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generation-gtfs-from-shapes-v2/2-generation-files-simples/../1-scraping_ruta_directa/data/proc/tampico.geojson
Shapes crudos (# rutas): (140, 10)


,ciudad_nombre,encodedLine,data.route.shortName,data.route.longName,slug,type,color,trips,shape_id,geometry,route_id
0,Tampico,u~qfCrcgtQxRnIf@gEoQ}HaIqDaAvDsCyAcRyHiYkM_WyK...,Mirador - Aviación por Boulevard,1_Mirador - Aviación por Boulevard,ruta-1-mirador-aviacion-por-boulevard.4uH,Bus,None,[ ],shape_3124,"LINESTRING (-97.85418 22.21563, -97.85586 22.2...",1
1,Tampico,}sigCr~ltQkF}CYr@]bGHf@hM`Ip@?Re@ZaGAY_GuDtIgF...,Tampico - Bosque Central de Abastos - Casa Blanca,2_Tampico - Bosque Central de Abastos - Casa B...,ruta-2-tampico-bosque-central-de-abastos-casa-...,Bus,None,[ ],shape_3125,"LINESTRING (-97.8841 22.33679, -97.88331 22.33...",2
2,Tampico,ssigCf~ltQ}F}CKv@_@tGZr@nMjHr@]VoHkGiDrIiF}C}G...,Madero - Revolución Verde - Central de Abastos,3_Madero - Revolución Verde - Central de Abastos,ruta-3-madero-revolucion-verde-central-de-abas...,Bus,None,[ ],shape_3126,"LINESTRING (-97.88404 22.33674, -97.88325 22.3...",3
3,Tampico,o`hgCdyetQbMuCjDbN~EbJtLb[j@V?l@xZwR`@YndAgV|A...,Santa Elena - Colonias - Tampico,9_Santa Elena - Colonias - Tampico,ruta-9-santa-elena-colonias-tampico.4uR,Bus,None,[ ],shape_3132,"LINESTRING (-97.84739 22.32856, -97.84664 22.3...",9
4,Tampico,ajggCz`ptQgB?rEoa@xObDrLzBjE~A|@?K_AfFiBU}AjBm...,Colosio - Águila Madero - Golfo,4_Colosio - Águila Madero - Golfo,ruta-4-colosio-aguila-madero-golfo.4uL,Bus,None,[ ],shape_3127,"LINESTRING (-97.89982 22.32497, -97.89982 22.3...",4


### Formato  y limpieza de rutas

A partir de `routes_raw` y los parámetros del notebook, se arma la tabla con columnas: `route_id | agency_id | route_short_name | route_long_name | route_type`.

In [6]:
routes_clean = routes_raw.copy()
routes_clean.drop(columns=["ciudad_nombre", "encodedLine", "data.route.longName", "slug", "shape_id", "color", "trips" ], inplace=True)
routes_clean = routes_clean[["route_id", "data.route.shortName", "type", "geometry"]]
routes_clean.head()

,route_id,data.route.shortName,type,geometry
0,1,Mirador - Aviación por Boulevard,Bus,"LINESTRING (-97.85418 22.21563, -97.85586 22.2..."
1,2,Tampico - Bosque Central de Abastos - Casa Blanca,Bus,"LINESTRING (-97.8841 22.33679, -97.88331 22.33..."
2,3,Madero - Revolución Verde - Central de Abastos,Bus,"LINESTRING (-97.88404 22.33674, -97.88325 22.3..."
3,9,Santa Elena - Colonias - Tampico,Bus,"LINESTRING (-97.84739 22.32856, -97.84664 22.3..."
4,4,Colosio - Águila Madero - Golfo,Bus,"LINESTRING (-97.89982 22.32497, -97.89982 22.3..."


### Fix duplicados
Verificar que no haya route id duplicados y si hay duplicados poner subincide _a, _b, _etc

In [7]:
duplicados_count = routes_clean["route_id"].duplicated().sum()
print(f"Total de route_id duplicados: {duplicados_count}")

# keep=False muestra todas las ocurrencias del duplicado, no solo la segunda
df_duplicados = routes_clean[routes_clean.duplicated(subset=["route_id"], keep=False)]
df_duplicados.sort_values("route_id")


Total de route_id duplicados: 1


,route_id,data.route.shortName,type,geometry
10,11,Puerto Alegre - Candelario Garza Tampico,Bus,"LINESTRING (-97.85316 22.21716, -97.85241 22.2..."
12,11,Tampico - Colonias - El Fuerte - Penal por Av....,Bus,"LINESTRING (-98.07474 22.43424, -98.07414 22.4..."


In [8]:
# 1. Identificamos qué filas están duplicadas (todas las ocurrencias)
es_duplicado = routes_clean.duplicated(subset=['route_id'], keep=False)

# 2. Generamos el sufijo (0 -> a, 1 -> b, etc.) usando el código ASCII
# cumcount() da 0 para la primera vez que ve un ID, 1 para la segunda, etc.
sufijos = routes_clean.groupby('route_id').cumcount().map(lambda x: f"_{chr(97 + x)}")

# 3. Aplicamos el cambio solo donde hay duplicados
routes_clean.loc[es_duplicado, 'route_id'] = routes_clean.loc[es_duplicado, 'route_id'].astype(str) + sufijos

Formateo de columnas

In [9]:
routes_clean.rename(columns={"data.route.shortName": "route_long_name", 
                                "type": "route_type"
                                }, inplace=True)

routes_clean["agency_id"] = AGENCY_ID

routes_clean["route_id"] = routes_clean["route_id"].str.upper()


routes_clean["route_id"] =  "Route_" + routes_clean["route_id"].astype(str)
routes_clean["route_short_name"] =  routes_clean["route_id"].astype(str)


routes_clean["shape_id"] = "Shape_" + routes_clean["route_id"]

routes_clean["route_type"] = routes_clean["route_type"].map(GTFS_ROUTE_TYPES).fillna(3).astype(int)

routes_clean = routes_clean[[ "route_id", "agency_id",  "route_short_name", "route_long_name", "route_type", "shape_id", "geometry"]].copy()
routes_clean.head(2)

,route_id,agency_id,route_short_name,route_long_name,route_type,shape_id,geometry
0,Route_1,IMEPLAN_Tampico,Route_1,Mirador - Aviación por Boulevard,3,Shape_Route_1,"LINESTRING (-97.85418 22.21563, -97.85586 22.2..."
1,Route_2,IMEPLAN_Tampico,Route_2,Tampico - Bosque Central de Abastos - Casa Blanca,3,Shape_Route_2,"LINESTRING (-97.8841 22.33679, -97.88331 22.33..."


Eliminar nans en geometry

In [10]:
# reproyectar
print(routes_clean.crs)  # debería decir EPSG:4326
routes_clean = routes_clean.to_crs(epsg=32614) # Reproyectar a UTM 14N (metros)

# Calcular longitud en metros
routes_clean["length_m"] = routes_clean.geometry.length
routes_clean.sort_values("length_m")

# Eliminar las filas donde length_m es NaN
routes_clean = routes_clean.dropna(subset=["length_m"])

EPSG:4326


/Users/danielbustillos/miniconda3/envs/analisis-general/lib/python3.10/site-packages/shapely/measurement.py:182: RuntimeWarning: invalid value encountered in length
  return lib.length(geometry, **kwargs)


In [11]:
routes_clean.drop(columns=["length_m"], inplace=True)
routes_clean.head()

,route_id,agency_id,route_short_name,route_long_name,route_type,shape_id,geometry
0,Route_1,IMEPLAN_Tampico,Route_1,Mirador - Aviación por Boulevard,3,Shape_Route_1,"LINESTRING (618098.532 2457141.016, 617928.009..."
1,Route_2,IMEPLAN_Tampico,Route_2,Tampico - Bosque Central de Abastos - Casa Blanca,3,Shape_Route_2,"LINESTRING (614915.384 2470530.991, 614995.778..."
2,Route_3,IMEPLAN_Tampico,Route_3,Madero - Revolución Verde - Central de Abastos,3,Shape_Route_3,"LINESTRING (614921.605 2470525.501, 615001.925..."
3,Route_9,IMEPLAN_Tampico,Route_9,Santa Elena - Colonias - Tampico,3,Shape_Route_9,"LINESTRING (618703.105 2469648.325, 618782.266..."
4,Route_4,IMEPLAN_Tampico,Route_4,Colosio - Águila Madero - Golfo,3,Shape_Route_4,"LINESTRING (613305.943 2469210.556, 613305.523..."


## 2. Formatear en formato GTFS routes

Formateamos la tabla y nos ajustamos a routes.txt, eliminamos la geometría

In [12]:
# A partir de routes_clean y parámetros (AGENCY_ID): tabla GTFS
df_routes_gtfs = pd.DataFrame({
    "route_id": routes_clean["route_id"],
    "agency_id": routes_clean["agency_id"],
    "route_short_name": routes_clean["route_id"],
    "route_long_name": routes_clean["route_long_name"],
    "route_type": routes_clean["route_type"],
})

df_routes_gtfs.head(5)

,route_id,agency_id,route_short_name,route_long_name,route_type
0,Route_1,IMEPLAN_Tampico,Route_1,Mirador - Aviación por Boulevard,3
1,Route_2,IMEPLAN_Tampico,Route_2,Tampico - Bosque Central de Abastos - Casa Blanca,3
2,Route_3,IMEPLAN_Tampico,Route_3,Madero - Revolución Verde - Central de Abastos,3
3,Route_9,IMEPLAN_Tampico,Route_9,Santa Elena - Colonias - Tampico,3
4,Route_4,IMEPLAN_Tampico,Route_4,Colosio - Águila Madero - Golfo,3


## Generación shapes.txt

In [13]:
# shapes.txt: shape_id, shape_pt_lat, shape_pt_lon, shape_pt_sequence [, shape_dist_traveled]
# routes_clean tiene geometry en UTM (32614). Reproyectamos a WGS84 para lat/lon.
from shapely.geometry import Point

routes_wgs84 = routes_clean.to_crs(epsg=4326)

filas = []
for idx in routes_clean.index:
    row_utm = routes_clean.loc[idx]
    row_wgs = routes_wgs84.loc[idx]
    shape_id = row_utm["shape_id"]
    geom_utm = row_utm.geometry
    geom_wgs = row_wgs.geometry
    if geom_wgs is None or geom_wgs.is_empty:
        continue
    coords_wgs = list(geom_wgs.coords)  # (lon, lat)
    # Distancia acumulada en km desde la geometría UTM
    if geom_utm is not None and not geom_utm.is_empty:
        coords_utm = list(geom_utm.coords)
        dist_accum = 0.0
        dists_km = [0.0]
        for i in range(1, len(coords_utm)):
            seg = Point(coords_utm[i - 1]).distance(Point(coords_utm[i]))  # metros
            dist_accum += seg / 1000.0  # km
            dists_km.append(round(dist_accum, 6))
    else:
        dists_km = [0.0] * len(coords_wgs)

    # Omitir puntos consecutivos duplicados (mismas coordenadas) para cumplir GTFS
    last_pt = None
    seq = 0
    for i, (lon, lat) in enumerate(coords_wgs):
        pt = (round(lat, 6), round(lon, 6))
        if pt == last_pt:
            continue
        last_pt = pt
        seq += 1
        filas.append({
            "shape_id": shape_id,
            "shape_pt_lat": pt[0],
            "shape_pt_lon": pt[1],
            "shape_pt_sequence": seq,
            "shape_dist_traveled": dists_km[i] if i < len(dists_km) else "",
        })

df_shapes = pd.DataFrame(filas)
df_shapes.head()

,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,Shape_Route_1,22.21563,-97.85418,1,0.000000
1,Shape_Route_1,22.21246,-97.85586,2,0.391348
2,Shape_Route_1,22.21226,-97.85486,3,0.496783
3,Shape_Route_1,22.21522,-97.85327,4,0.863182
4,Shape_Route_1,22.21683,-97.85238,5,1.063647


### Validación shapes

Comprueba que para cada `shape_id`: (1) `shape_dist_traveled` sea siempre ascendente según `shape_pt_sequence`, y (2) no haya puntos con la misma latitud y longitud (duplicados).

In [14]:
def validar_shapes_gtfs(df):
    """
    Valida que en un DataFrame de shapes (GTFS):
    - shape_dist_traveled sea ascendente (no decreciente) para cada shape_id según shape_pt_sequence.
    - No haya puntos con la misma (lat, lon) consecutivos (duplicados).
    Devuelve True si todo es válido, False si hay errores. Imprime un resumen.
    """
    if df.empty or "shape_id" not in df.columns:
        print("DataFrame vacío o sin columna shape_id.")
        return False

    req = ["shape_id", "shape_pt_lat", "shape_pt_lon", "shape_pt_sequence", "shape_dist_traveled"]
    missing = [c for c in req if c not in df.columns]
    if missing:
        print(f"Faltan columnas: {missing}")
        return False

    df = df.sort_values(["shape_id", "shape_pt_sequence"]).reset_index(drop=True)
    errores_dist = []
    errores_dup = []

    for shape_id, grp in df.groupby("shape_id", sort=False):
        grp = grp.sort_values("shape_pt_sequence")
        dist = grp["shape_dist_traveled"]
        lat = grp["shape_pt_lat"].round(6)
        lon = grp["shape_pt_lon"].round(6)

        # 1) shape_dist_traveled ascendente (cada valor >= anterior)
        if pd.api.types.is_numeric_dtype(dist):
            for i in range(1, len(dist)):
                if dist.iloc[i] < dist.iloc[i - 1]:
                    errores_dist.append((shape_id, int(grp["shape_pt_sequence"].iloc[i]), dist.iloc[i - 1], dist.iloc[i]))
        # Si hay vacíos/string, no validamos dist

        # 2) No (lat, lon) consecutivos duplicados
        for i in range(1, len(grp)):
            if (lat.iloc[i] == lat.iloc[i - 1]) and (lon.iloc[i] == lon.iloc[i - 1]):
                errores_dup.append((shape_id, int(grp["shape_pt_sequence"].iloc[i]), lat.iloc[i], lon.iloc[i]))

    ok = len(errores_dist) == 0 and len(errores_dup) == 0
    if errores_dist:
        print(f"[Validación] shape_dist_traveled no ascendente: {len(errores_dist)} caso(s). Ejemplos: {errores_dist[:5]}")
    if errores_dup:
        print(f"[Validación] (lat, lon) consecutivos duplicados: {len(errores_dup)} caso(s). Ejemplos: {errores_dup[:5]}")
    if ok:
        print("[Validación] OK: shape_dist_traveled ascendente por shape_id y sin (lat, lon) consecutivos duplicados.")
    return ok


validar_shapes_gtfs(df_shapes)

[Validación] OK: shape_dist_traveled ascendente por shape_id y sin (lat, lon) consecutivos duplicados.


True

## Export files

In [15]:
# Unimos la carpeta con el nombre del archivo
ruta_final_routes_gtfs = PATH_DIR_GTFS / "routes.txt"
df_routes_gtfs.to_csv(ruta_final_routes_gtfs, index=False) # archivo rutas GTFS

In [16]:
# Unimos la carpeta con el nombre del archivo
ruta_final_processed_routes = PATH_DIR_proccesed / "routes_clean.geojson"
# Guardamos directamente
routes_clean.to_file(ruta_final_processed_routes, driver="GeoJSON")

In [17]:
# Exportar a GTFS
path_shapes = PATH_DIR_GTFS / "shapes.txt"
df_shapes.to_csv(path_shapes, index=False, encoding="utf-8")
print(f"Guardado: {path_shapes} ({len(df_shapes)} puntos, {df_shapes['shape_id'].nunique()} shapes)")

df_shapes.head(10)

Guardado: ../data/tampico/gtfs-output/shapes.txt (12801 puntos, 138 shapes)


,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,Shape_Route_1,22.21563,-97.85418,1,0.000000
1,Shape_Route_1,22.21246,-97.85586,2,0.391348
2,Shape_Route_1,22.21226,-97.85486,3,0.496783
3,Shape_Route_1,22.21522,-97.85327,4,0.863182
4,Shape_Route_1,22.21683,-97.85238,5,1.063647
5,Shape_Route_1,22.21716,-97.85330,6,1.165275
6,Shape_Route_1,22.21790,-97.85285,7,1.259420
7,Shape_Route_1,22.22096,-97.85128,8,1.634857
8,Shape_Route_1,22.22517,-97.84898,9,2.157769
9,Shape_Route_1,22.22901,-97.84693,10,2.632506
